<h1>
  Pricing options and computing implied volatilities using Neural Networks
</h1>

---

### **Importing Necessary Libraries**

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import norm
from joblib import dump, load
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

Found GPU at: /device:GPU:0


## **Dataset Generation**

### Black Scholes Equation

In [3]:
'''
Black-Scholes Equation for Option Pricing
 '''
def Black_Scholes(S, K, T, r, sigma):
  d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
  d2 = d1 - sigma * np.sqrt(T)
  Call_Prices = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
  return Call_Prices

In [4]:
'''
Sample Size
'''
Sample_Size = 1000000


'''
Set the random seed for reproducibility
'''
np.random.seed(42)


'''
Generate random values for the parameters
'''
S = np.random.uniform(50, 800, Sample_Size)
K = np.random.uniform(120, 500, Sample_Size)
T = np.random.uniform(0.2, 1.1, Sample_Size)
r = np.random.uniform(0.02, 0.1, Sample_Size)
sigma = np.random.uniform(0.01, 1.0, Sample_Size)


'''
Calculate the call option prices using the Black-Scholes equation
'''
call_prices = Black_Scholes(S, K, T, r, sigma)


'''
Create a DataFrame to store the dataset
'''
df = pd.DataFrame({
    'Stock Price (S)': S,
    'Strike Price (K)': K,
    'Time to Maturity (T)': T,
    'Risk-Free Rate (r)': r,
    'Volatility (sigma)': sigma,
    'Call Option Price': call_prices
})


'''
Save the dataset to a CSV file
'''
df.to_csv('options_data.csv', index=False)

##  **Artifitial Neural Network**

*Table 2. The Selected Model After the Random Search*

<div>
    <style>
        table {
            border-collapse: collapse;
            width: 50%;
            margin: 20px auto;
            font-family: Arial, sans-serif;
        }
        th, td {
            border: 1px solid #ddd;
            padding: 8px;
            text-align: left;
        }
        th {
            background-color: #f2f2f2;
        }
    </style>
    <table>
        <thead>
            <tr>
                <th>Parameter</th>
                <th>Option</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Hidden layers</td>
                <td>4</td>
            </tr>
            <tr>
                <td>Neurons (each layer)</td>
                <td>400</td>
            </tr>
            <tr>
                <td>Activation</td>
                <td>ReLU</td>
            </tr>
            <tr>
                <td>Dropout rate</td>
                <td>0.0</td>
            </tr>
            <tr>
                <td>Batch-normalization</td>
                <td>No</td>
            </tr>
            <tr>
                <td>Initialization</td>
                <td>Glorot_uniform</td>
            </tr>
            <tr>
                <td>Optimizer</td>
                <td>Adam</td>
            </tr>
            <tr>
                <td>Batch size</td>
                <td>1024</td>
            </tr>
        </tbody>
    </table>
</div>

In [5]:
'''
Initialization of Artifitial Neural Network
'''
model = models.Sequential()


'''
Input layer {
    S: Current stock price.
    K: Strike price.
    T: Time to maturity.
    r: Risk-free interest rate.
    σ: Volatility of the stock's returns.
}
'''

model.add(layers.Input(shape=(5,)))


'''
4 Hidden Layers for each 400 neurons and Initialized the Weight using Glorot_uniform for better convergence.
'''
model.add(layers.Dense(400, activation='relu', kernel_initializer='glorot_uniform'))
model.add(layers.Dense(400, activation='relu', kernel_initializer='glorot_uniform'))
model.add(layers.Dense(400, activation='relu', kernel_initializer='glorot_uniform'))
model.add(layers.Dense(400, activation='relu', kernel_initializer='glorot_uniform'))


'''
Output Layer: The output of the model is the Option Price
'''
model.add(layers.Dense(1, activation='linear'))

In [8]:
'''
Defining the Optimizer for the Model
 '''
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(optimizer=optimizer, loss='mean_squared_error',metrics=['mean_absolute_error'])

### **Training and Testing**

In [9]:
'''
Importing Created Dataset
'''
Dataset = pd.read_csv('/content/options_data.csv')

In [10]:
'''
Separating into input and output variables
'''
y = Dataset['Call Option Price'].values
X = Dataset.drop('Call Option Price', axis=1).values

In [11]:
'''
Splitting the dataset into the Training set and Test set
'''
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.02, random_state=42)

In [12]:
'''
Feature Scaling for better convergence
'''
scaller = StandardScaler()
X_train = scaller.fit_transform(X_train)
X_test = scaller.transform(X_test)

In [26]:
'''
Saving Standard Scaler for future use
'''
dump(scaller, 'scaler.joblib')

['scaler.joblib']

In [27]:
'''
Importing Standard Scaler
'''
scaler = load('scaler.joblib')

In [14]:
'''
Training the Model
 '''
history = model.fit(X_train, y_train,
                        batch_size=1024,
                        epochs=100,
                        verbose=1,
                        validation_split=0.2)

Epoch 1/100
766/766 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - loss: 7675.0928 - mean_absolute_error: 34.5090 - val_loss: 2.0116 - val_mean_absolute_error: 1.0520
Epoch 2/100
766/766 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 1.4411 - mean_absolute_error: 0.8863 - val_loss: 1.3175 - val_mean_absolute_error: 0.8788
Epoch 3/100
766/766 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 1.0724 - mean_absolute_error: 0.7651 - val_loss: 0.6503 - val_mean_absolute_error: 0.5955
Epoch 4/100
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.1977 - mean_absolute_error: 0.7689 - val_loss: 0.4462 - val_mean_absolute_error: 0.4871
Epoch 5/100
766/766 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.5682 - mean_absolute_error: 0.8275 - val_loss: 2.6881 - val_mean_absolute_error: 1.2672
Epoch 6/100
766/766 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 2.0836 - mean_absolute_error: 0.9643 - val_loss: 1.4992 - val_mean_absolute_error: 0.9725
Epoch 7/100
766/766 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1.5546 - mean_absolute_error:

In [16]:
'''
Save the model for future use
'''
model.save('/content/Option_Pricing_Model.keras')

### Model Evaluation

In [33]:
'''
Prediction on Test Set
 '''
y_pred = model.predict(X_test)

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [34]:
'''
 Creating DataFrame for better visualization
'''
y_pred = y_pred.flatten()
results_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

In [41]:
'''
Setting the output format to 6 decimal places
 '''
pd.options.display.float_format = '{:.6f}'.format

In [42]:
'''
Printing the first 20 rows of the DataFrame
 '''
print(results_df.head(20))

       Actual  Predicted
0   43.408340  43.197906
1   78.423806  78.330231
2   65.122344  64.876083
3  313.528592 313.128357
4  209.165269 208.559219
5  545.694425 544.757996
6   95.334467  95.136536
7  283.602129 283.000732
8    0.000000   0.000093
9   92.533395  92.180702
10 114.867816 114.551315
11 359.557978 358.804626
12 134.319315 134.019226
13 636.464661 635.038513
14   0.000022   0.009965
15 103.944267 103.714142
16 123.865656 123.554062
17 167.625031 167.253326
18 349.261093 348.355164
19  92.687806  92.393921


In [43]:
'''
The Mean Average Error
'''
mse = np.mean(np.abs(y_test - y_pred))
print(f"Mean Average Error: {mse}")

Mean Average Error: 0.4259690994508196
